In [ ]:
import pandas as pd
from sklearn.datasets import load_wine

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. CLASS DISTRIBUTION (Target Variable)
# Checks if the dataset is balanced (How many Malignant vs Benign?)
plt.figure(figsize=(6, 4))
sns.countplot(x=y, palette="viridis")
plt.title("Target Class Distribution (0: Malignant, 1: Benign)")
plt.xlabel("Class")
plt.ylabel("Count")
plt.show()

# 2. CORRELATION HEATMAP
# Shows how features are related to each other. Dark red/blue means high correlation.
plt.figure(figsize=(10, 8))
# annot=False because printing numbers for a 30x30 grid would be unreadable
sns.heatmap(X.corr(), annot=False, cmap="coolwarm") 
plt.title("Feature Correlation Heatmap")
plt.show()

# 3. BOXPLOT (Outlier Detection)
# We plot only the first 5 features, otherwise the chart gets too crowded.
plt.figure(figsize=(10, 6))
sns.boxplot(data=X.iloc[:, :5], palette="Set2")
plt.title("Outlier Check: Boxplot of First 5 Features")
plt.show()

In [ ]:
X, y = load_wine(return_X_y=True, as_frame=True) # as_frame part returns as pandas database
print("Missing data count:")
print(X.isnull().sum().sum())

display(X.head()) # first 5 elements of X

In [ ]:
from sklearn.model_selection import train_test_split
X_remain, X_test, y_remain, y_test = train_test_split(X, y, test_size = 0.2, stratify=y, random_state=42) #20% test, hide


In [ ]:
seeds = [42, 52, 62]
divided_data = {}

for seed in seeds:
    X_train, X_confirm, y_train, y_confirm = train_test_split(X_remain, y_remain, test_size=0.125, stratify=y_remain, random_state=seed)

    divided_data[seed] = {  # put values to dict on current seed
        "X_train": X_train, "X_confirm" : X_confirm,
        "y_train": y_train, "y_confirm" : y_confirm,
    }

In [ ]:
from sklearn.preprocessing import StandardScaler

for seed in seeds:
    X_train = divided_data[seed]["X_train"]
    X_confirm = divided_data[seed]["X_confirm"]

    scaler = StandardScaler() # scale them (train data and validating one)
    X_train_scaled = scaler.fit_transform(X_train)
    X_confirm_scaled = scaler.transform(X_confirm)

    divided_data[seed]["X_train_scaled"] = X_train_scaled         # put scaled values to dict
    divided_data[seed]["X_confirm_scaled"] = X_confirm_scaled
    divided_data[seed]["scaler"] = scaler

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
import pandas as pd

model_results = []
dt_depths = [2, 3, 5, 10, None]
knn_neighbours = [1, 3, 5, 7, 11]

for seed in seeds:
    for scale in ["unscaled", "scaled"]:
        if scale == "unscaled":
            X_train = divided_data[seed]["X_train"]
            X_confirm = divided_data[seed]["X_confirm"]
        elif scale == "scaled":
            X_train = divided_data[seed]["X_train_scaled"]
            X_confirm = divided_data[seed]["X_confirm_scaled"]

        y_train = divided_data[seed]["y_train"]
        y_confirm = divided_data[seed]["y_confirm"]

        

        def evaluatemodel(model, algorithm_name, param_name):
            model.fit(X_train, y_train)

            train_predict = model.predict(X_train)
            confirm_predict = model.predict(X_confirm)
            confirm_proba = model.predict_proba(X_confirm)

            train_accuracy = accuracy_score(y_train, train_predict)    #calculate for train set
            train_f1 = f1_score(y_train, train_predict, average="macro", zero_division = 0)

            confirm_acc = accuracy_score(y_confirm, confirm_predict)   #calculate for confirm set
            confirm_prec = precision_score(y_confirm, confirm_predict, average="macro", zero_division=0)
            confirm_recall = recall_score(y_confirm, confirm_predict, average="macro", zero_division=0)
            confirm_f1 = f1_score(y_confirm, confirm_predict, average="macro", zero_division=0)
            confirm_roc = roc_auc_score(y_confirm, confirm_proba, multi_class="ovr")

            run_id = f"Wine_{algorithm_name}_{scale}_{param_name}_s{seed}"
            
            model_results.append({ #appending results to model_results kist
                "Run_ID": run_id, "Dataset": "Wine", "Seed": seed, 
                "Algorithm": algorithm_name, "Representation": scale, "Parameter": param_name,
                "Train_Accuracy": train_accuracy, "Train_Macro_F1": train_f1,
                "Val_Accuracy": confirm_acc, "Val_Macro_Precision": confirm_prec, 
                "Val_Macro_Recall": confirm_recall, "Val_Macro_F1": confirm_f1, "Val_ROC_AUC": confirm_roc
            })

        
        # 1 decision tree loop
        for depth in dt_depths:
            # create model, assign seed to random state
            dt_model = DecisionTreeClassifier(max_depth=depth, random_state=seed)
            # Asistana gönder
            evaluatemodel(dt_model, "dt", f"md{depth}")
            
        # 2 KNN loop
        for neighbour in knn_neighbours:
            # create model
            knn_model = KNeighborsClassifier(n_neighbors=neighbour, weights='uniform', metric='euclidean')
            # send to assistnat
            evaluatemodel(knn_model, "knn", f"k{neighbour}")
            
        # 3 NAIVE BAYES
        gnb_model = GaussianNB()
        evaluatemodel(gnb_model, "gnb", "default")


# turn results to dataframe
df_results = pd.DataFrame(model_results)

print(f"Total models ran: {len(df_results)}")   #show first 5 results
display(df_results.head())


In [ ]:
# mean
summary_table = df_results.groupby(
    ['Algorithm', 'Representation', 'Parameter']
).agg(
    Mean_Val_Macro_F1=('Val_Macro_F1', 'mean'),
    Std_Val_Macro_F1=('Val_Macro_F1', lambda x: x.std(ddof=1)),
    Mean_Val_Accuracy=('Val_Accuracy', 'mean')
).reset_index()


summary_table = summary_table.sort_values(
    by=['Mean_Val_Macro_F1', 'Mean_Val_Accuracy', 'Std_Val_Macro_F1'],
    ascending=[False, False, True] # descend descend ascend
)

# choose best model on top of table
best_model = summary_table.iloc[0]

print("22 ROW SUMMARY TABLE (Top 5)")
display(summary_table.head())

print("\n BEST MODEL:")
print(f"Algorithm: {best_model['Algorithm']}, Scale: {best_model['Representation']}, Setting: {best_model['Parameter']}")

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# 1. PREPARE FINAL DATA (Strictly using Seed 42)
if best_model['Representation'] == 'scaled':
    X_train_final = divided_data[42]["X_train_scaled"]
    # CRITICAL: Only transform the test set using Seed 42's scaler. NEVER fit!
    X_test_final = divided_data[42]["scaler"].transform(X_test)
else:
    X_train_final = divided_data[42]["X_train"]
    X_test_final = X_test

y_train_final = divided_data[42]["y_train"]

# 2. INITIALIZE CHAMPION MODEL
if best_model['Algorithm'] == 'dt':
    depth_val = best_model['Parameter'].replace('md', '')
    depth = int(depth_val) if depth_val != 'None' else None
    final_model = DecisionTreeClassifier(max_depth=depth, random_state=42)
    
elif best_model['Algorithm'] == 'knn':
    neighbors = int(best_model['Parameter'].replace('k', ''))
    final_model = KNeighborsClassifier(n_neighbors=neighbors, weights='uniform', metric='euclidean')
    
elif best_model['Algorithm'] == 'gnb':
    final_model = GaussianNB()

# 3. TRAIN AND PREDICT
final_model.fit(X_train_final, y_train_final)

test_predictions = final_model.predict(X_test_final)
test_probabilities = final_model.predict_proba(X_test_final)[:, 0]

# 4. CALCULATE FINAL METRICS
test_acc = accuracy_score(y_test, test_predictions)
test_f1 = f1_score(y_test, test_predictions, average="macro")

y_test_inverted = 1 - y_test # Make Malignant (0) the positive class for ROC-AUC
test_roc = roc_auc_score(y_test_inverted, test_probabilities)

print("--- FINAL TEST METRICS ---")
print(f"Accuracy: {test_acc:.4f} | Macro-F1: {test_f1:.4f} | ROC-AUC: {test_roc:.4f}\n")

print("--- CLASSIFICATION REPORT ---")
print(classification_report(y_test, test_predictions, target_names=["Malignant (0)", "Benign (1)"]))

# 5. CONFUSION MATRIX
cm = confusion_matrix(y_test, test_predictions, labels=[0, 1])

plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=["Malignant (0)", "Benign (1)"], 
            yticklabels=["Malignant (0)", "Benign (1)"])
plt.ylabel('Actual Label')
plt.xlabel('Predicted Label')
plt.title('Confusion Matrix - Final Test')
plt.show()

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, rand_score
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns

# 1. SCALE THE ENTIRE DATASET 
# Unsupervised learning uses the whole dataset without labels.
scaler_kmeans = StandardScaler()
X_full_scaled = scaler_kmeans.fit_transform(X)

# 2. INITIALIZE AND RUN K-MEANS
# We use 2 clusters because we expect Malignant and Benign groups
kmeans = KMeans(n_clusters=3, n_init=10, random_state=42)
cluster_labels = kmeans.fit_predict(X_full_scaled) # Notice we only give X, not y!

# 3. METRICS
# Rand Index: How well do the clusters match the actual hidden labels (y)?
rand_idx = rand_score(y, cluster_labels)
# Silhouette Score: How distinct/tight are the clusters?
silhouette = silhouette_score(X_full_scaled, cluster_labels)

print("K-MEANS UNSUPERVISED LEARNING RESULTS")
print(f"Rand Index: {rand_idx:.4f}")
print(f"Silhouette Score: {silhouette:.4f}\n")

# 4. VISUALIZATION (Plotting the first 2 features as a sample)
plt.figure(figsize=(7,5))
sns.scatterplot(x=X_full_scaled[:, 0], y=X_full_scaled[:, 1], hue=cluster_labels, palette="viridis")
plt.title("K-Means Clusters (Feature 0 vs Feature 1)")
plt.xlabel("Feature 0 (Scaled)")
plt.ylabel("Feature 1 (Scaled)")
plt.show()